In [3]:
# execute as if in root folder for utils and filepaths
%cd ..

c:\Users\sonja\OneDrive\Uni\Semester 6\BachelorArbeit\practical-work-ai-bsc


C:\Users\sonja\AppData\Roaming\Python\Python311\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset
import numpy as np
import random
import warnings
import pandas as pd
import json
from torch.utils.tensorboard import SummaryWriter
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, auc, precision_recall_curve , average_precision_score


from utils import make_support_query, log_results

In [5]:
data = np.load('./data/data_scaled.npz')
X_finetuning = data['X_finetuning']
y_ft1 = data['y_finetuning1']
y_ft2 = data['y_finetuning2']

In [7]:
seeds = [8479, 227, 5413, 8179, 7528]

In [8]:
# -------------- RF ------------- #

# list to collect results per task
all_results = {
        "Delta-AUC-PR": [],
        "ROC-AUC": []
    }  
# list to collect results avg over task
avg_results = {
        "Delta-AUC-PR": [],
        "ROC-AUC": []
    }

# train RF five times and avg results
for run in range(5):
    results = []
    for i in range(4):
        # get support and query set
        S, Q = make_support_query(X_finetuning, y_ft1, task_idx=i)
        X_train, y_train = S
        X_test, y_test = Q

        # remove samples with Nan test
        mask_valid_test = ~np.isnan(y_test)
        X_test_valid = X_test[mask_valid_test]
        y_test_valid = y_test[mask_valid_test]
        
        # fraction of positive samples in train set (-> how rare is label 1 vs label 0)
        mask_pos_train = (y_test_valid == 1)
        percent_pos_train = len(y_test_valid[mask_pos_train]) / len(y_test_valid)
        
        # rf 
        rf = RandomForestClassifier(random_state=seeds[run])
        rf.fit(X_train, y_train)
        
        # predictions & metrics
        y_pred_proba = rf.predict_proba(X_test_valid)[:, 1]
        auc = roc_auc_score(y_test_valid, y_pred_proba)
        delta_auc_pr = average_precision_score(y_test_valid, y_pred_proba) - percent_pos_train
        
        # factor: better or worse than random: <1: worse
        factor = delta_auc_pr / percent_pos_train
        factor = round(factor, 2)
        
        # results dict
        results.append({
            "Task": i + 1,
            "Pos. Labels/Train": round(percent_pos_train, 6),
            "ROC-AUC": round(auc, 4),
            "Delta-AUC-PR": round(delta_auc_pr, 6),
            "Factor Better/Worse": factor
        })

        df_results = pd.DataFrame(results)
    
    # collect results for all tasks
    all_results["Delta-AUC-PR"].append(np.array(df_results["Delta-AUC-PR"]))
    all_results["ROC-AUC"].append(np.array(df_results["ROC-AUC"]))

    # collect results avg over tasks
    mean_auc_pr = np.mean(np.array(df_results["Delta-AUC-PR"]))
    mean_roc_auc = np.mean(np.array(df_results["ROC-AUC"]))
    avg_results["Delta-AUC-PR"].append(mean_auc_pr)
    avg_results["ROC-AUC"].append(mean_roc_auc)

In [9]:
filepath_normal = "./metrics/metrics_baseline_ft1.json"
filepath_per_task = "./metrics/metrics_baseline_ft1_per_task.json"
filepath_per_run = "./metrics/metrics_baseline_ft1_per_run.json"
log_results(avg_results, all_results, filepath_normal, filepath_per_run, filepath_per_task)

In [ ]:
# # finetuning loop
# for epoch in num_epochs:
#     for episode in num_episodes_per_epoch:
#         t = random_task
#         S, Q = support and query set for task
        
#         # update step etc